# Competition for compounds from other members: consumption overlap

In [1]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network

In [2]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")

In [3]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

In [4]:
communities_12_exp_abundance = pd.read_csv("../data/Venturelli2018/generated/communities_12_exp_abundance.tsv",sep="\t",index_col=[0,1])

In [5]:
m9_mets = list(media_db[media_db.medium=="M9[-O2]"].compound.values)
m9_mets.remove("so4")
m9_mets.remove("nh4")
m9_mets.remove("glc__D")

m9_mets = ["M_"+met+"__91__e__93__" for met in m9_mets]

In [6]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]

In [7]:
results = results[~results.compound.isin(m9_mets)].copy()

In [8]:
def centrality_metrics(network_data,cutoff=0,SP_centrality=True):
    """
    SP_centrality: centrality meassures based on shortest path
    """
    # Betweenness and closeness
    G = nx.Graph()
    G.add_nodes_from(list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique()))))
    
    if SP_centrality:
        edges = [(index[0],index[1],{"weight":1/data} ) for index,data in network_data.items() if data>cutoff]
    
    else:
        edges = [(index[0],index[1],{"weight":data} ) for index,data in network_data.items() if data>cutoff]
        
    G.add_edges_from(edges)
    
    if SP_centrality:
        return pd.DataFrame({"betweeness (1/data)":nx.centrality.betweenness_centrality(G,weight="weight"),
                     "closeness (1/data)":nx.centrality.closeness_centrality(G,distance="weight")})
    else:
        return pd.DataFrame({"degree (data)":dict(G.degree(weight="weight")),
                 "eigenvector (data)":nx.centrality.eigenvector_centrality(G,weight="weight")})


In [9]:
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)


In [10]:
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue

In [11]:
def spearman_corr_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*10,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [12]:
def count_overlap_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network()
    
    maximum_centrality = pd.Series(closeness_centrality_met).max()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    
    
    color = [colorFader(b1,b2,closeness_centrality_met[node]/maximum_centrality) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)

    maximum_overlap = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)/10,color=colorFader(c1,c2,value/maximum_overlap))

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

### Spearman correlation

In [13]:
results_env = results[(results.donor!="environment") & (results.receiver!="environment")]
results_env = results_env[["receiver","compound","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env.groupby(["receiver","compound"]).mean()

**Correlation matrix**

In [14]:
network_data = results_grouped_rec_env.unstack().T.corr(method="spearman")
network_data = network_data.mask(np.tril(np.ones(network_data.shape)).astype(np.bool)).stack() 

In [15]:
network_data

receiver                                receiver                              
Bacteroides_ovatus_ATCC_8483            Bacteroides_thetaiotaomicron_VPI_5482     0.592564
                                        Bacteroides_uniformis_ATCC_8492           0.375032
                                        Bacteroides_vulgatus_ATCC_8482            0.467769
                                        Blautia_hydrogenotrophica_DSM_10507       0.718578
                                        Clostridium_hiranonis_TO_931_DSM_13275    0.333333
                                        Collinsella_aerofaciens_ATCC_25986        0.423053
                                        Desulfovibrio_piger_ATCC_29098            0.131075
                                        Eggerthella_lenta_DSM_2243                0.457552
                                        Eubacterium_rectale_ATCC_33656            0.732308
                                        Faecalibacterium_prausnitzii_A2_165       0.458930
Bacteroides

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [16]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_ovatus_ATCC_8483,0.000000,0.458760
Eggerthella_lenta_DSM_2243,0.000000,0.508993
Desulfovibrio_piger_ATCC_29098,0.000000,0.527256
Collinsella_aerofaciens_ATCC_25986,0.022222,0.527503
Eubacterium_rectale_ATCC_33656,0.000000,0.541716
Bacteroides_vulgatus_ATCC_8482,0.000000,0.547532
Clostridium_hiranonis_TO_931_DSM_13275,0.022222,0.553934
Bacteroides_uniformis_ATCC_8492,0.000000,0.555602
Faecalibacterium_prausnitzii_A2_165,0.022222,0.573111
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.661310


In [17]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Bacteroides_ovatus_ATCC_8483,4.690195,0.245758
Eggerthella_lenta_DSM_2243,5.238669,0.270693
Desulfovibrio_piger_ATCC_29098,5.483950,0.285004
Collinsella_aerofaciens_ATCC_25986,5.655289,0.289491
Bacteroides_vulgatus_ATCC_8482,5.669273,0.291451
Eubacterium_rectale_ATCC_33656,5.753025,0.293497
Clostridium_hiranonis_TO_931_DSM_13275,6.006347,0.309353
Faecalibacterium_prausnitzii_A2_165,6.054138,0.310897
Bacteroides_uniformis_ATCC_8492,6.137021,0.316484
Bacteroides_thetaiotaomicron_VPI_5482,6.766672,0.341807


In [18]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [19]:
net = spearman_corr_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


**Network topological properties (frequency>0.9!!!)**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [20]:
cutoff = 0.9

In [21]:
centrality_metrics_SP = centrality_metrics(network_data,cutoff=cutoff).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_vulgatus_ATCC_8482,0.0,0.000000
Bacteroides_uniformis_ATCC_8492,0.0,0.000000
Clostridium_hiranonis_TO_931_DSM_13275,0.0,0.000000
Bacteroides_thetaiotaomicron_VPI_5482,0.0,0.000000
Blautia_hydrogenotrophica_DSM_10507,0.0,0.000000
Eggerthella_lenta_DSM_2243,0.0,0.000000
Bacteroides_ovatus_ATCC_8483,0.0,0.000000
Eubacterium_rectale_ATCC_33656,0.0,0.000000
Faecalibacterium_prausnitzii_A2_165,0.0,0.000000
Collinsella_aerofaciens_ATCC_25986,0.0,0.090862


In [22]:
centrality_metrics(network_data,SP_centrality=False,cutoff=cutoff).sort_values("degree (data)")

,degree (data),eigenvector (data)
Bacteroides_vulgatus_ATCC_8482,0.000000,9.002601e-07
Bacteroides_uniformis_ATCC_8492,0.000000,9.002601e-07
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,9.002601e-07
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,9.002601e-07
Blautia_hydrogenotrophica_DSM_10507,0.000000,9.002601e-07
Eggerthella_lenta_DSM_2243,0.000000,9.002601e-07
Bacteroides_ovatus_ATCC_8483,0.000000,9.002601e-07
Eubacterium_rectale_ATCC_33656,0.000000,9.002601e-07
Faecalibacterium_prausnitzii_A2_165,0.000000,9.002601e-07
Collinsella_aerofaciens_ATCC_25986,0.908622,7.071068e-01


**Network (cutoff correlation = 0.9 !!)**

In [23]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [24]:
net = spearman_corr_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=cutoff)
net.show("example.html")

example.html


### Overlap count (compound consumed with high frequency)

In [25]:
results_env = results[(results.donor!="environment") & (results.receiver!="environment")]
results_env = results_env[["receiver","compound","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env.groupby(["receiver","compound"]).mean()

**Count (cutoff frequency=0.5!!)**

In [26]:
count_same = results_grouped_rec_env["frequency"]

count_same = count_same.unstack().fillna(0)>0.1
count_same = count_same.astype(int)
count_same = count_same.dot(count_same.T)

network_data = count_same.mask(np.tril(np.ones(count_same.shape)).astype(np.bool)).stack() 

maximum = network_data.max()

In [27]:
results_grouped_rec_env.sort_values("frequency").tail(40)

,,frequency
receiver,compound,
Desulfovibrio_piger_ATCC_29098,M_acald__91__e__93__,0.213250
Eggerthella_lenta_DSM_2243,M_12dhchol__91__e__93__,0.214000
Bacteroides_ovatus_ATCC_8483,M_nh4__91__e__93__,0.216556
Bacteroides_thetaiotaomicron_VPI_5482,M_7ocholate__91__e__93__,0.227000
Bacteroides_uniformis_ATCC_8492,M_adn__91__e__93__,0.237500
Blautia_hydrogenotrophica_DSM_10507,M_fru__91__e__93__,0.244000
Clostridium_hiranonis_TO_931_DSM_13275,M_adn__91__e__93__,0.260333
Bacteroides_vulgatus_ATCC_8482,M_adn__91__e__93__,0.260667
Collinsella_aerofaciens_ATCC_25986,M_fru__91__e__93__,0.288500


**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [28]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Faecalibacterium_prausnitzii_A2_165,0.000000,1.538462
Desulfovibrio_piger_ATCC_29098,0.000000,1.764706
Bacteroides_uniformis_ATCC_8492,0.000000,1.834862
Eggerthella_lenta_DSM_2243,0.010000,1.954397
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,2.127660
Bacteroides_ovatus_ATCC_8483,0.000000,2.142857
Collinsella_aerofaciens_ATCC_25986,0.062593,2.352941
Bacteroides_thetaiotaomicron_VPI_5482,0.029630,2.553191
Eubacterium_rectale_ATCC_33656,0.030370,2.690583
Blautia_hydrogenotrophica_DSM_10507,0.087778,2.727273


In [29]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Desulfovibrio_piger_ATCC_29098,14.0,0.177338
Faecalibacterium_prausnitzii_A2_165,16.0,0.187288
Eggerthella_lenta_DSM_2243,18.0,0.229410
Bacteroides_uniformis_ATCC_8492,19.0,0.247184
Clostridium_hiranonis_TO_931_DSM_13275,23.0,0.295209
Bacteroides_ovatus_ATCC_8483,23.0,0.300691
Collinsella_aerofaciens_ATCC_25986,25.0,0.294576
Eubacterium_rectale_ATCC_33656,28.0,0.344127
Blautia_hydrogenotrophica_DSM_10507,29.0,0.346066
Bacteroides_thetaiotaomicron_VPI_5482,30.0,0.374559


**Network**

In [30]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=0)
net.show("example.html")

example.html


**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [31]:
centrality_metrics_SP = centrality_metrics(network_data,cutoff=2).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Faecalibacterium_prausnitzii_A2_165,0.000000,1.318681
Eggerthella_lenta_DSM_2243,0.000000,1.648352
Desulfovibrio_piger_ATCC_29098,0.000000,1.685393
Bacteroides_uniformis_ATCC_8492,0.000000,1.769912
Bacteroides_ovatus_ATCC_8483,0.000000,2.033898
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,2.054795
Collinsella_aerofaciens_ATCC_25986,0.200000,2.181818
Blautia_hydrogenotrophica_DSM_10507,0.044444,2.500000
Bacteroides_thetaiotaomicron_VPI_5482,0.088889,2.553191
Eubacterium_rectale_ATCC_33656,0.116667,2.631579


In [32]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Desulfovibrio_piger_ATCC_29098,14.0,0.177338
Faecalibacterium_prausnitzii_A2_165,16.0,0.187288
Eggerthella_lenta_DSM_2243,18.0,0.229410
Bacteroides_uniformis_ATCC_8492,19.0,0.247184
Clostridium_hiranonis_TO_931_DSM_13275,23.0,0.295209
Bacteroides_ovatus_ATCC_8483,23.0,0.300691
Collinsella_aerofaciens_ATCC_25986,25.0,0.294576
Eubacterium_rectale_ATCC_33656,28.0,0.344127
Blautia_hydrogenotrophica_DSM_10507,29.0,0.346066
Bacteroides_thetaiotaomicron_VPI_5482,30.0,0.374559


**Network**

In [33]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=2)
net.show("example.html")

example.html
